# ⚡ AegisX — QLoRA Fine-Tune (power path, free T4)

When AegisX-Mini (from scratch) hits its ceiling, this notebook gives it real
power: fine-tunes a small open model (**Qwen2.5-3B**) on your cybersecurity
instruction data with **QLoRA** (4-bit, runs on a free T4).

Output: a **merged full model saved to Drive** for manual Hugging Face upload
(same manual flow as AegisX-Mini — no auto-push).

## 1. Setup

In [ ]:
!pip install -q torch transformers accelerate peft bitsandbytes trl datasets
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Mount Drive (output survives disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/aegisx/finetuned'
!mkdir -p {OUT_DIR}

## 3. Build an instruction dataset

Format: `instruction` / `output` rows. You can build this from your raw corpus
with an LLM later, or start with a hand-written seed file. Cells below load
`data/finetune/instructions.jsonl` if present, else create a small seed.

In [ ]:
import json
from pathlib import Path

data_file = Path('data/finetune/instructions.jsonl')
if data_file.exists():
    rows = [json.loads(l) for l in data_file.read_text().splitlines() if l.strip()]
    print(f'Loaded {len(rows)} instruction rows from {data_file}')
else:
    rows = [
        {'instruction': 'How do I prevent SQL injection?', 'output': 'Use parameterized queries so user input is never concatenated into SQL. Validate and sanitize input server-side.'},
        {'instruction': 'What is a race condition in bug bounty?', 'output': 'Two operations run concurrently and the app fails to enforce a limit, e.g. spending the same gift card twice. Test with parallel requests.'},
        {'instruction': 'How do I enumerate subdomains?', 'output': 'Use certificate transparency logs (crt.sh), DNS brute force with wordlists, and check dangling DNS records.'},
        {'instruction': 'What should a bug bounty report include?', 'output': 'Vulnerability type, affected endpoint, reproduction steps, impact, and a suggested fix. Be honest about severity.'},
    ]
    print(f'Using {len(rows)} seed rows. Add your own to data/finetune/instructions.jsonl')

In [ ]:
from datasets import Dataset

def fmt(row):
    return {'text': f"### Instruction\n{row['instruction']}\n\n### Response\n{row['output']}"}

ds = Dataset.from_list([fmt(r) for r in rows])
print(ds)

## 4. Load base model in 4-bit + tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = 'Qwen/Qwen2.5-3B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model.config.use_cache = False
print('base model loaded (4-bit)')

## 5. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=SFTConfig(
        output_dir='/content/aegisx-lora',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_steps=200,
        warmup_steps=20,
        logging_steps=10,
        save_steps=50,
        bf16=True,
    ),
    dataset_text_field='text',
    max_seq_length=1024,
)
trainer.train()
print('training done')

## 7. Merge LoRA into the base and save FULL model

Merging produces a standalone model (no adapter needed at inference) that
you can upload to Hugging Face manually like AegisX-Mini.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import shutil

# Reload base in fp16 (merge needs full precision, not 4-bit)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True)
merged = PeftModel.from_pretrained(base, '/content/aegisx-lora')
merged = merged.merge_and_unload()

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tok.pad_token = tok.eos_token

final_dir = OUT_DIR + '/aegisx-qwen-3b'
if Path(final_dir).exists():
    shutil.rmtree(final_dir)
merged.save_pretrained(final_dir, safe_serialization=True)
tok.save_pretrained(final_dir)
print('Merged model saved to', final_dir)

## 8. Sanity check

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

m = AutoModelForCausalLM.from_pretrained(final_dir, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
t = AutoTokenizer.from_pretrained(final_dir, trust_remote_code=True)
prompt = '### Instruction\nWhat is the first step of a penetration test?\n\n### Response\n'
ids = t(prompt, return_tensors='pt').to('cuda')
out = m.generate(**ids, max_new_tokens=80, do_sample=False)
print(t.decode(out[0], skip_special_tokens=True))

## 9. Upload to Hugging Face (manual)

Download `MyDrive/aegisx/finetuned/aegisx-qwen-3b/` from Drive, then:

1. https://huggingface.co/new - **Model**, name `aegisx-qwen-3b`
2. **Files > Add file > Upload files** - drop the whole folder contents
   (this is a full model: `model.safetensors`, `config.json`, `tokenizer*`, ...)
3. Commit - it's a standard Transformers model, so Spaces/`pipeline()` work
   out of the box: `pipeline('text-generation', model='FerzDevZ/aegisx-qwen-3b')`